# 2ème partie du projet

In [ ]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from mistralai.client import Mistral

from dotenv import load_dotenv
import os
load_dotenv(Path.cwd() / ".env")
api_key = os.environ["MISTRAL_API_KEY"]

In [ ]:
# Création de la base Faiss et de l'index
# FAISS ne stocke que les vecteurs. On sauvegarde donc l’index FAISS
#  et un Parquet de métadonnées avec le même faiss_id.

# Colonnes à retourner avec les résultats de recherche.
colonnes_metadata_candidates = [
    "score_similarite",
    "uid",
    "title",
    "texte_chunk",
    "nextTiming",
    "lastTiming",
    "timings",
    "location.name",
    "location.city",
    "location.region",
    "location.address",
    "agenda_titre_source",
]

df_embeddings = pd.read_parquet("data/parquet_sortie/evenements_embeddings_mistral.parquet")
df_nettoye = pd.read_parquet("data/parquet_sortie/evenements_culturels_nettoye.parquet")

colonnes_metadata = [
    colonne
    for colonne in colonnes_metadata_candidates
    if colonne in df_nettoye.columns
]

# Traitement des métadonnées
# Associe chaque chunk à l'événement dont il provient.
metadata_evenements = df_nettoye[colonnes_metadata].copy()
metadata_evenements["index_evenement"] = metadata_evenements.index

# Un chunk correspond à un seul evt mais plusieurs chunks peuvent correspondre à un même evt.
metadata_faiss = (
    df_embeddings[["index_evenement", "numero_chunk", "texte_chunk"]]
    .merge(
        metadata_evenements,
        on="index_evenement",
        how="left",
        validate="many_to_one",
    )
    .reset_index(drop=True)
)

# Les IDs FAISS sont stables et correspondent aux lignes des métadonnées.
# Un faiss_id unique pour chaque chunk, pour retrouver les métadonnées après la recherche.
metadata_faiss.insert(
    0,
    "faiss_id",
    np.arange(len(metadata_faiss), dtype=np.int64),
)

# Matrice de vecteurs de type float32.
# FAISS exige une matrice NumPy en float32.
vecteurs = np.asarray(
    df_embeddings["embedding"].tolist(),
    dtype=np.float32,
)

if vecteurs.ndim != 2 or len(vecteurs) != len(metadata_faiss):
    raise ValueError("Incohérence entre les embeddings et les métadonnées.")

# normalisation L2 = similarité cosinus
faiss.normalize_L2(vecteurs)

# Recherche de similarité cosinus avec FAISS par produit scalaire des 2 vecteurs
dimension = vecteurs.shape[1]
index_faiss = faiss.IndexIDMap2(faiss.IndexFlatIP(dimension))

index_faiss.add_with_ids(
    vecteurs,
    metadata_faiss["faiss_id"].to_numpy(dtype=np.int64),
)

# Sauvegardes locales
dossier_index = Path("data/index_faiss")
dossier_index.mkdir(parents=True, exist_ok=True)

# vecteurs et index de recherche
faiss.write_index(index_faiss, str(dossier_index / "evenements.faiss"))

# métadonnées associées aux vecteurs (chunks, titres, lieux, dates et identifiants)
metadata_faiss.drop(columns="embedding", errors="ignore").to_parquet(
    dossier_index / "metadata.parquet",
    index=False,
)

print(f"{index_faiss.ntotal} chunks indexés en dimension {dimension}.")

# Tests des index et de la recherche

In [ ]:
# Tous les embeddings et métadata ont été ajoutés à FAISS.
assert index_faiss.ntotal == len(vecteurs)
assert index_faiss.ntotal == len(metadata_faiss)

# Chaque chunk possède un identifiant FAISS unique.
assert metadata_faiss["faiss_id"].is_unique

# Chaque événement ayant généré un embedding est présent dans les métadonnées.
ids_attendus = set(df_embeddings["index_evenement"].unique())
ids_indexes = set(metadata_faiss["index_evenement"].unique())

ids_manquants = ids_attendus - ids_indexes
assert not ids_manquants, f"Événements non indexés : {ids_manquants}"

print(
    f"Index valide : {index_faiss.ntotal} chunks indexés, "
    f"{len(ids_indexes)} événements couverts."
)

In [ ]:
def rechercher_réponse(question, k=5):
    with Mistral(api_key=api_key) as mistral:
        reponse = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[question],
        )

    vecteur_question = np.asarray(
        [reponse.data[0].embedding],
        dtype=np.float32,
    )
    # On normalise le vecteur de la question pour la recherche par similarité cosinus.
    faiss.normalize_L2(vecteur_question)

    # Faiss retourne id des k chunks les plus proches et leur score de similarité (produit scalaire des vecteurs normalisés = cosinus).
    scores, ids = index_faiss.search(vecteur_question, k)

    # Les id trouvés sont traduits en métadonnées pour retourner les informations de l'événement correspondant.
    resultats = metadata_faiss.set_index("faiss_id").loc[ids[0]].copy()
    resultats["score_similarite"] = scores[0]

    return resultats[
        [
        "score_similarite",
        "uid",
        "title",
        "texte_chunk",
        "timings",
        "location.name",
        "location.city",
        "location.region",
        "location.address",
        "agenda_titre_source",
        ]
    ]

In [ ]:
print(metadata_faiss.columns.tolist())

In [ ]:
resultats = rechercher_réponse(
    "Je cherche un match de football à Quimper ce week-end",
    k=5,
)

display(resultats)

# Solution 1 : intégration de la recherche dans LangChain

In [ ]:
from langchain_core.documents import Document
from mistralai.client import Mistral
import numpy as np
import faiss


def rechercher_evenements(question, k=5):
    """
    Recherche sémantique dans FAISS et retourne des Documents LangChain.
    """

    # 1. Création de l'embedding de la question
    with Mistral(api_key=api_key) as mistral:
        response = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[question],
        )

    vecteur_question = np.asarray(
        [response.data[0].embedding],
        dtype=np.float32,
    )

    # 2. Normalisation pour similarité cosinus
    faiss.normalize_L2(vecteur_question)

    # 3. Recherche FAISS
    scores, ids = index_faiss.search(
        vecteur_question,
        k
    )

    documents = []

    for score, faiss_id in zip(scores[0], ids[0]):

        # -1 signifie généralement qu'aucun résultat n'a été trouvé
        if faiss_id == -1:
            continue

        # Récupération des métadonnées
        ligne = (
            metadata_faiss
            .loc[metadata_faiss["faiss_id"] == faiss_id]
            .iloc[0]
        )

        # Construction du contenu transmis au LLM
        contenu = f"""
            Titre : {ligne["title"]}

            Description :
            {ligne["texte_chunk"]}

            Horaires :
            {ligne["timings"]}

            Lieu :
            {ligne["location.name"]}

            Ville :
            {ligne["location.city"]}

            Région :
            {ligne["location.region"]}

            Adresse :
            {ligne["location.address"]}
        """

        document = Document(
            page_content=contenu,
            metadata={
                "uid": ligne["uid"],
                "title": ligne["title"],
                "faiss_id": int(faiss_id),
                "score_similarite": float(score),
                "agenda_titre_source": ligne["agenda_titre_source"],
            }
        )

        documents.append(document)

    return documents

In [ ]:
# On transforme la recherche sémantique en outil LangChain pour l'utiliser dans un agent.

from langchain_core.tools import tool


@tool
def rechercher_evenements_culturels(question: str) -> str:
    """
    Recherche des événements culturels dans la base d'événements.
    """

    print("\n=== APPEL OUTIL FAISS ===")
    print("Question :", question)

    documents = rechercher_evenements(
        question=question,
        k=5
    )

    print("Nombre de documents :", len(documents))

    if not documents:
        print("Aucun document trouvé.")
        return "Aucun événement pertinent trouvé."

    textes = []

    for doc in documents:

        print(
            "Document trouvé :",
            doc.metadata["title"]
        )

        texte = f"""
Titre : {doc.metadata["title"]}

Score : {doc.metadata["score_similarite"]:.3f}

{doc.page_content}
"""

        textes.append(texte.strip())

    print("=== FIN APPEL OUTIL FAISS ===\n")

    return "\n\n---\n\n".join(textes)

In [ ]:
# On construit un agent qui fait appel à l'outil Faiss si besoin
from gc import set_debug

from langchain.agents import create_agent
from langchain_mistralai import ChatMistralAI



# Création du modèle de langage
llm = ChatMistralAI(
    model_name="mistral-small-latest",
    temperature=0.2,
    timeout=10,
    max_retries=0,
)

tools = [
    rechercher_evenements_culturels
]


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
      Tu es un assistant spécialisé dans les événements culturels.

      Tu disposes d'un outil permettant de rechercher des événements dans
      une base de données.

      Règles :

      1. Utilise l'outil de recherche lorsqu'une question concerne des
        événements culturels présents dans la base.

      2. Pour les questions générales ne nécessitant pas les données de la
        base, réponds directement.

      3. N'invente jamais un événement, une date, un horaire, un lieu ou une
        information qui ne figure pas dans les résultats de recherche.

      4. Lorsque plusieurs événements sont trouvés, privilégie ceux qui
        correspondent le mieux à la demande de l'utilisateur.

      5. Si aucun résultat pertinent n'est trouvé, indique-le clairement
        plutôt que d'inventer une réponse.

      6. Les expressions temporelles telles que "aujourd'hui", "ce soir",
        "demain", "ce week-end", "la semaine prochaine", etc. doivent être
        considérées comme des contraintes importantes de la question.

      7. Ne considère pas qu'un résultat est pertinent uniquement parce que
        sa description est sémantiquement proche de la question : les
        contraintes explicites de lieu, de date ou de catégorie doivent
        également être respectées lorsqu'elles peuvent être vérifiées.

      8. Réponds de manière claire et concise à l'utilisateur.
    """
)

In [ ]:
# Appel à Faiss sans passer par l'agent LangChain
resultat = rechercher_evenements_culturels.invoke(
    {
        "question": "Quels concerts sont prévus ce week-end ?"
    }
)

print(resultat)

In [ ]:
from mistralai.client import Mistral

client = Mistral()

tools = [
    {
        "type": "function",
        "function": {
            "name": "test_tool",
            "description": "Recherche des événements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Question de l'utilisateur."
                    }
                },
                "required": ["question"]
            }
        }
    }
]

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {
            "role": "user",
            "content": "Quels concerts sont prévus ce week-end ?"
        }
    ],
    tools=tools,
    tool_choice="any",
)

print(response)

In [ ]:
# Tests avec une question sur un événement culturel
question = "Quels concerts sont prévus ce week-end à Vannes ?"

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(response["messages"][-1].content)

# Solution 2 : Recherche avec appel Mistral directement

In [ ]:
import os
import numpy as np
import faiss

from mistralai.client import Mistral

# ============================================================
# 1. Configuration
# ============================================================

# La clé API est déjà présente dans les variables d'environnement.
# Le SDK Mistral la récupère automatiquement.
client = Mistral()


# ============================================================
# 2. Recherche sémantique dans FAISS
# ============================================================

def rechercher_semantique_mistral(question, k=5):
    """
    Recherche les événements les plus proches sémantiquement
    de la question dans l'index FAISS.
    """

    # Création de l'embedding de la question
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[question],
    )

    vecteur_question = np.asarray(
        [response.data[0].embedding],
        dtype=np.float32,
    )

    # Normalisation pour utiliser la similarité cosinus
    faiss.normalize_L2(vecteur_question)

    # Recherche dans FAISS
    scores, ids = index_faiss.search(
        vecteur_question,
        k,
    )

    # Récupération des métadonnées
    resultats = (
        metadata_faiss
        .set_index("faiss_id")
        .loc[ids[0]]
        .copy()
    )

    resultats["score_similarite"] = scores[0]

    return resultats[
        [
            "score_similarite",
            "uid",
            "title",
            "texte_chunk",
            "timings",
            "location.name",
            "location.city",
            "location.region",
            "location.address",
            "agenda_titre_source",
        ]
    ]


# ============================================================
# 3. Fonction appelée par Mistral
# ============================================================

def rechercher_evenements(question, k=5):
    """
    Effectue une recherche sémantique dans la base FAISS
    et transforme les résultats en texte exploitable par Mistral.
    """

    resultats = rechercher_semantique_mistral(
        question,
        k=5,
    )

    if resultats.empty:
        return "Aucun événement correspondant n'a été trouvé."

    textes = []

    for _, evenement in resultats.iterrows():

        texte = f"""
Titre : {evenement["title"]}

Description :
{evenement["texte_chunk"]}

Horaires :
{evenement["timings"]}

Lieu :
{evenement["location.name"]}

Ville :
{evenement["location.city"]}

Région :
{evenement["location.region"]}

Adresse :
{evenement["location.address"]}
"""

        textes.append(texte.strip())

    return "\n\n---\n\n".join(textes)


# ============================================================
# 4. Définition du tool pour Mistral
# ============================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "rechercher_evenements",
            "description": (
                "Recherche dans la base des événements culturels "
                "correspondant à la question de l'utilisateur. "
                "Utiliser cette fonction lorsque la question porte "
                "sur des événements, concerts, spectacles, festivals, "
                "expositions, activités, lieux, dates ou horaires."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": (
                            "Question de l'utilisateur à utiliser "
                            "pour effectuer la recherche."
                        ),
                    }
                },
                "required": ["question"],
            },
        },
    }
]


# ============================================================
# 5. Fonction chatbot
# ============================================================

def chatbot(question):
    """
    Envoie une question à Mistral.

    Mistral décide s'il doit utiliser la recherche FAISS.
    Si oui, le résultat de FAISS est renvoyé à Mistral
    afin qu'il formule la réponse finale.
    """

    messages = [
        {
            "role": "system",
            "content": (
                """
                Tu es un assistant spécialisé dans les événements culturels.

                Tu disposes d'un outil permettant de rechercher des événements dans
                une base de données.

                Règles :

                1. Utilise l'outil de recherche lorsqu'une question concerne des
                événements culturels présents dans la base.

                2. Pour les questions générales ne nécessitant pas les données de la
                base, réponds directement.

                3. N'invente jamais un événement, une date, un horaire, un lieu ou une
                information qui ne figure pas dans les résultats de recherche.

                4. Lorsque plusieurs événements sont trouvés, privilégie ceux qui
                correspondent le mieux à la demande de l'utilisateur.

                5. Si aucun résultat pertinent n'est trouvé, indique-le clairement
                plutôt que d'inventer une réponse.

                6. Les expressions temporelles telles que "aujourd'hui", "ce soir",
                "demain", "ce week-end", "la semaine prochaine", etc. doivent être
                considérées comme des contraintes importantes de la question.

                7. Ne considère pas qu'un résultat est pertinent uniquement parce que
                sa description est sémantiquement proche de la question : les
                contraintes explicites de lieu, de date ou de catégorie doivent
                également être respectées lorsqu'elles peuvent être vérifiées.

                8. Réponds de manière claire et concise à l'utilisateur.
            """
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    # --------------------------------------------------------
    # Premier appel : Mistral analyse la question
    # --------------------------------------------------------

    response = client.chat.complete(
        model="mistral-small-latest",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    message = response.choices[0].message

    # Ajout de la réponse de Mistral à l'historique
    messages.append(message)

    # --------------------------------------------------------
    # Mistral demande-t-il l'utilisation du tool ?
    # --------------------------------------------------------

    if not message.tool_calls:

        # Pas besoin de FAISS :
        return message.content

    # --------------------------------------------------------
    # Exécution des tools demandés par Mistral
    # --------------------------------------------------------

    for tool_call in message.tool_calls:

        if tool_call.function.name == "rechercher_evenements":

            arguments = tool_call.function.arguments

            # Selon la version du SDK, arguments peut être
            # déjà un dictionnaire ou une chaîne JSON.
            if isinstance(arguments, str):
                import json
                arguments = json.loads(arguments)

            resultat = rechercher_evenements(
                arguments["question"]
            )

            # Résultat du tool ajouté à la conversation
            messages.append(
                {
                    "role": "tool",
                    "name": tool_call.function.name,
                    "content": resultat,
                    "tool_call_id": tool_call.id,
                }
            )

    # --------------------------------------------------------
    # Deuxième appel : Mistral formule la réponse finale
    # --------------------------------------------------------

    final_response = client.chat.complete(
        model="mistral-small-latest",
        messages=messages,
    )

    return final_response.choices[0].message.content


In [ ]:
# Tests

question = "Quels concerts sont prévus ce week-end à Vannes ?"

reponse = chatbot(question)

print("QUESTION :")
print(question)

print("\nRÉPONSE :")
print(reponse)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss

from sklearn.decomposition import PCA
from mistralai.client import Mistral


# ============================================================
# 1. Question à analyser
# ============================================================

question = "Quels concerts sont prévus à Vannes ?"


# ============================================================
# 2. Récupération de l'embedding de la question
# ============================================================

client = Mistral(api_key=api_key)

response = client.embeddings.create(
    model="mistral-embed",
    inputs=[question],
)

vecteur_question = np.asarray(
    [response.data[0].embedding],
    dtype=np.float32,
)


# ============================================================
# 3. Recherche des 5 voisins dans FAISS
# ============================================================

vecteur_question_normalise = vecteur_question.copy()
faiss.normalize_L2(vecteur_question_normalise)

scores, ids = index_faiss.search(
    vecteur_question_normalise,
    5
)

ids = ids[0]
scores = scores[0]


# ============================================================
# 4. Récupération des métadonnées des résultats
# ============================================================

resultats = (
    metadata_faiss[
        metadata_faiss["faiss_id"].isin(ids)
    ]
    .copy()
)

# Remet les résultats dans l'ordre FAISS
ordre = {faiss_id: rang for rang, faiss_id in enumerate(ids)}
resultats["ordre"] = resultats["faiss_id"].map(ordre)
resultats["score_similarite"] = (
    resultats["faiss_id"].map(
        dict(zip(ids, scores))
    )
)

resultats = (
    resultats
    .sort_values("ordre")
    .reset_index(drop=True)
)

print("Résultats FAISS :\n")

for i, ligne in resultats.iterrows():

    print(
        f"{i+1}. {ligne['title']} "
        f"(score = {ligne['score_similarite']:.3f})"
    )


# ============================================================
# 5. Préparation des embeddings
# ============================================================

# La colonne 'embedding' contient les vecteurs 1024D.
X = np.vstack(
    df_embeddings["embedding"].tolist()
).astype(np.float32)

print("\nNombre d'embeddings :", X.shape[0])
print("Dimension :", X.shape[1])


# ============================================================
# 6. PCA sur tous les embeddings
# ============================================================

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X)

print("\nVariance expliquée :")
print(
    f"PC1 : {pca.explained_variance_ratio_[0]:.2%}"
)
print(
    f"PC2 : {pca.explained_variance_ratio_[1]:.2%}"
)
print(
    f"Total : "
    f"{pca.explained_variance_ratio_.sum():.2%}"
)


# ============================================================
# 7. Projection de la question
# ============================================================

question_pca = pca.transform(
    vecteur_question
)


# ============================================================
# 8. Retrouver les lignes df_embeddings correspondant
#    aux résultats FAISS
# ============================================================

cles = resultats[
    [
        "faiss_id",
        "index_evenement",
        "numero_chunk",
        "title",
        "score_similarite",
    ]
].copy()

embeddings_resultats = df_embeddings.merge(
    cles,
    on=["index_evenement", "numero_chunk"],
    how="inner",
)

print(
    "\nNombre de résultats FAISS retrouvés dans df_embeddings :",
    len(embeddings_resultats)
)


# ============================================================
# 9. Projection des résultats FAISS
# ============================================================

X_resultats = np.vstack(
    embeddings_resultats["embedding"].tolist()
).astype(np.float32)

resultats_pca = pca.transform(X_resultats)


# ============================================================
# 10. Visualisation
# ============================================================

plt.figure(figsize=(12, 8))

# Tous les embeddings
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    alpha=0.15,
    s=15,
    label="Tous les embeddings",
)

# Résultats FAISS
plt.scatter(
    resultats_pca[:, 0],
    resultats_pca[:, 1],
    s=90,
    label="Résultats FAISS",
)

# Question
plt.scatter(
    question_pca[0, 0],
    question_pca[0, 1],
    marker="*",
    s=250,
    label="Question",
)


# ============================================================
# 11. Annotation des résultats
# ============================================================

for i, (_, ligne) in enumerate(
    embeddings_resultats.iterrows()
):
    plt.annotate(
        ligne["title"],
        (
            resultats_pca[i, 0],
            resultats_pca[i, 1],
        ),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )


plt.xlabel("Composante principale 1")
plt.ylabel("Composante principale 2")

plt.title(
    f'Projection PCA — « {question} »'
)

plt.legend()
plt.tight_layout()
plt.show()

# Tests des différents scenarii

In [ ]:
import time

questions_test = [
    {
        "id": 1,
        "scenario": "Recherche simple",
        "question": "Quels concerts sont prévus à Vannes ?",
    },
    {
        "id": 2,
        "scenario": "Recherche avec lieu",
        "question": "Quels événements sont prévus à Lorient ?",
    },
    {
        "id": 3,
        "scenario": "Recherche avec date relative",
        "question": "Quels concerts sont prévus ce week-end à Vannes ?",
    },
    {
        "id": 4,
        "scenario": "Recherche descriptive",
        "question": "Je cherche un spectacle de musique classique à Vannes.",
    },
    {
        "id": 5,
        "scenario": "Question générale",
        "question": "Qu'est-ce qu'un festival culturel ?",
    },
    {
        "id": 6,
        "scenario": "Aucun résultat probable",
        "question": "Quels événements de jazz sont prévus à Reykjavik ?",
    },
]


def tester_agent(question):
    """
    Envoie une question à l'agent et retourne sa réponse finale.
    """

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    return response["messages"][-1].content


for test in questions_test:

    print("=" * 80)
    print(f"TEST {test['id']} — {test['scenario']}")
    print("=" * 80)

    print(f"\nQuestion : {test['question']}")

    try:
        reponse = tester_agent(test["question"])

        print(f"\nRéponse :\n{reponse}")

    except Exception as e:
        print(f"\nERREUR : {type(e).__name__}")
        print(e)

    time.sleep(10)

    print()

In [ ]:
# Vérification du test 3
print(metadata_faiss.loc[
    metadata_faiss["location.city"].astype(str).str.lower().eq("vannes"),
    ["uid", "title", "timings"]
].head(20).to_string(index=False))

# Construction de l'API du RAG Voir api.py et les services